In [20]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


In [6]:
# to load llm credentials
load_dotenv()

True

In [5]:
model = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2
        )

In [14]:
# Define the state structure
class StateSchema(TypedDict):
    user_input: str
    response: str

In [15]:
def get_response(state:StateSchema) -> StateSchema:
    prompt = f'Answer the following question {state['user_input']}'
    response = model.invoke(prompt)
    state['response'] = response.content
    return state


In [16]:
#create Linear Graph
graph = StateGraph(StateSchema)
# create graph nodes
graph.add_node('get_answer', get_response)

# add edges
graph.add_edge(START, 'get_answer')
graph.add_edge('get_answer', END)

In [19]:
#compile
workflow = graph.compile()
#run workflow
initial_state = {'user_input': 'What is the capital of France?'}
output_state = workflow.invoke(initial_state)
output_state['response']

'The capital of France is **Paris**.'